# K-Means

K-Means clustering on California housing features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

target = data_df['median_house_value'].to_numpy()
raw_lon = data_df['longitude'].to_numpy()
raw_lat = data_df['latitude'].to_numpy()
X = data_df.drop(columns=['median_house_value']).to_numpy()

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(X.shape)

## Baseline (k=5)

In [ ]:
km = KMeans(n_clusters=5, n_init=10, random_state=42)
labels = km.fit_predict(X)

print(f"Inertia: {km.inertia_:.2f}")
print(f"Silhouette: {silhouette_score(X, labels, sample_size=5000, random_state=42):.4f}")
print("Cluster sizes:", np.bincount(labels))

## Elbow and silhouette

In [ ]:
ks = list(range(2, 11))
inertias, sils = [], []

for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(X, m.labels_, sample_size=5000, random_state=42))
    print(f"k={k:<2} inertia={m.inertia_:.0f} silhouette={sils[-1]:.4f}")

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].plot(ks, inertias, 'o-')
axs[0].set_xlabel('k')
axs[0].set_ylabel('inertia')
axs[0].set_title('elbow')
axs[0].grid(alpha=0.3)

axs[1].plot(ks, sils, 'o-', color='coral')
axs[1].set_xlabel('k')
axs[1].set_ylabel('silhouette')
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## PCA projection of clusters

In [ ]:
Z = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(10, 7))
for c in np.unique(labels):
    mask = labels == c
    plt.scatter(Z[mask, 0], Z[mask, 1], s=5, alpha=0.5, label=f'C{c}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=3)
plt.grid(alpha=0.3)
plt.show()

## Geographic view

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(raw_lon, raw_lat, c=labels, cmap='tab10', s=6, alpha=0.5)
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.grid(alpha=0.3)
plt.show()

## Target vs cluster

In [ ]:
plt.figure(figsize=(10, 5))
plt.boxplot([target[labels == c] for c in np.unique(labels)],
            tick_labels=[f'C{c}' for c in np.unique(labels)])
plt.ylabel('median_house_value')
plt.grid(alpha=0.3)
plt.show()

for c in np.unique(labels):
    vals = target[labels == c]
    print(f"C{c} n={len(vals):<6} mean={vals.mean():.0f} median={np.median(vals):.0f}")

## From-scratch K-Means

In [ ]:
class KMeansScratch:
    def __init__(self, k=5, max_iter=100, seed=42):
        self.k = k
        self.max_iter = max_iter
        self.seed = seed

    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        idx = rng.choice(len(X), self.k, replace=False)
        self.centroids = X[idx].copy()
        self.inertia_hist = []
        for _ in range(self.max_iter):
            d = np.linalg.norm(X[:, None] - self.centroids[None], axis=2)
            labels = d.argmin(axis=1)
            new_centroids = np.array([
                X[labels == c].mean(axis=0) if (labels == c).any() else self.centroids[c]
                for c in range(self.k)
            ])
            inertia = np.sum(np.min(d, axis=1) ** 2)
            self.inertia_hist.append(inertia)
            if np.allclose(new_centroids, self.centroids):
                break
            self.centroids = new_centroids
        self.labels_ = labels
        self.inertia_ = inertia
        return self

rng = np.random.default_rng(42)
sub = rng.choice(len(X), 3000, replace=False)
scratch = KMeansScratch(k=5).fit(X[sub])
print(f"Scratch inertia: {scratch.inertia_:.2f}")
print(f"Iters: {len(scratch.inertia_hist)}")

plt.figure(figsize=(8, 4))
plt.plot(scratch.inertia_hist, 'o-')
plt.xlabel('iter')
plt.ylabel('inertia')
plt.grid(alpha=0.3)
plt.show()